# Learning Rate Schedules - Theory Notebook

This notebook is the interactive companion to `notes.md`. It treats a learning-rate schedule as a mathematical function $\eta_t$ and studies how warmup, decay, cosine annealing, cyclic policies, one-cycle training, and WSD change optimization behavior.

The goal is not to memorize API names. The goal is to see the shape, compute the values, and diagnose the training loop when the schedule is wrong.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", palette="colorblind")
    HAS_SNS = True
except ImportError:
    plt.style.use("seaborn-v0_8-whitegrid")
    HAS_SNS = False

mpl.rcParams.update({
    "figure.figsize":    (10, 6),
    "figure.dpi":         120,
    "font.size":           13,
    "axes.titlesize":      15,
    "axes.labelsize":      13,
    "xtick.labelsize":     11,
    "ytick.labelsize":     11,
    "legend.fontsize":     11,
    "legend.framealpha":   0.85,
    "lines.linewidth":      2.0,
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "savefig.bbox":       "tight",
    "savefig.dpi":         150,
})
np.random.seed(42)
print("Plot setup complete.")

COLORS = {
    "primary":   "#0077BB",
    "secondary": "#EE7733",
    "tertiary":  "#009988",
    "error":     "#CC3311",
    "neutral":   "#555555",
    "highlight": "#EE3377",
}

def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    if not ok:
        print("  got     :", got)
        print("  expected:", expected)
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} - {name}")
    return cond


---

## 1. Fixed Learning Rates and Stability

For the quadratic $f(\boldsymbol{\theta})=\frac{1}{2}\boldsymbol{\theta}^{\top}H\boldsymbol{\theta}$, gradient descent is stable in the steepest eigen-direction only when $\eta < 2/\lambda_{\max}(H)$. This makes the fixed learning rate a compromise between fast movement and stability.


In [ ]:
# === 1.1 Fixed learning rate stability on a quadratic ===
H = np.diag([1.0, 60.0])
theta0 = np.array([4.0, 4.0])
eta_safe = 1.8 / np.max(np.diag(H))
eta_unsafe = 2.2 / np.max(np.diag(H))

def gd_path(theta, eta, steps=60):
    path = [theta.copy()]
    for _ in range(steps):
        grad = H @ theta
        theta = theta - eta * grad
        path.append(theta.copy())
    return np.array(path)

safe_path = gd_path(theta0.copy(), eta_safe)
unsafe_path = gd_path(theta0.copy(), eta_unsafe)

print(f"lambda_max = {np.max(np.diag(H)):.1f}")
print(f"safe eta   = {eta_safe:.5f}")
print(f"unsafe eta = {eta_unsafe:.5f}")
print(f"safe final norm   = {np.linalg.norm(safe_path[-1]):.6f}")
print(f"unsafe final norm = {np.linalg.norm(unsafe_path[-1]):.6f}")
check_true("safe path contracts", np.linalg.norm(safe_path[-1]) < np.linalg.norm(theta0))
check_true("unsafe path grows in steep direction", abs(unsafe_path[-1, 1]) > abs(theta0[1]))

fig, ax = plt.subplots()
ax.plot(np.linalg.norm(safe_path, axis=1), color=COLORS["primary"], label="stable eta")
ax.plot(np.linalg.norm(unsafe_path, axis=1), color=COLORS["error"], label="too-large eta")
ax.set_title("Fixed learning rate can be stable or divergent")
ax.set_xlabel("Step")
ax.set_ylabel(r"$\Vert \theta_t \Vert_2$")
ax.legend()
fig.tight_layout()
plt.show()


---

## 2. Schedule Functions

A schedule is a function $\eta_t=\eta_{\max}s(t)$. We will implement the common $s(t)$ shapes and verify their boundary behavior.


In [ ]:
# === 2.1 Schedule primitives ===
def constant_schedule(t, eta=1.0):
    t = np.asarray(t)
    return np.full_like(t, eta, dtype=float)

def step_decay(t, eta=1.0, drop_every=20, gamma=0.5):
    t = np.asarray(t)
    return eta * gamma ** (t // drop_every)

def exponential_decay(t, eta=1.0, gamma=0.98):
    t = np.asarray(t)
    return eta * gamma ** t

def polynomial_decay(t, eta=1.0, total_steps=100, power=1.0, eta_min=0.0):
    t = np.asarray(t)
    u = np.clip(t / total_steps, 0.0, 1.0)
    return eta_min + (eta - eta_min) * (1.0 - u) ** power

t = np.arange(0, 101)
print("constant first/last:", constant_schedule(t, 0.1)[0], constant_schedule(t, 0.1)[-1])
print("step values at 0,20,40:", step_decay(np.array([0, 20, 40]), eta=0.1))
print("exp value at 50:", exponential_decay(np.array([50]), eta=0.1, gamma=0.98)[0])
print("linear polynomial final:", polynomial_decay(np.array([100]), eta=0.1, total_steps=100)[0])
check_close("polynomial reaches eta_min", polynomial_decay(np.array([100]), eta=0.1, total_steps=100, eta_min=0.001), np.array([0.001]))


---

## 3. Phase Anatomy

Most modern schedules combine warmup, plateau or high-LR movement, and decay. Thinking in phases makes implementation bugs easier to spot.


In [ ]:
# === 3.1 Warmup-stable-decay phase function ===
def warmup_stable_decay(t, eta_max=1.0, warmup=10, stable=60, total=100, eta_min=0.05):
    t = np.asarray(t, dtype=float)
    lr = np.empty_like(t)
    warm_mask = t < warmup
    stable_mask = (t >= warmup) & (t < stable)
    decay_mask = t >= stable
    lr[warm_mask] = eta_max * t[warm_mask] / max(warmup, 1)
    lr[stable_mask] = eta_max
    u = np.clip((t[decay_mask] - stable) / max(total - stable, 1), 0.0, 1.0)
    lr[decay_mask] = eta_min + 0.5 * (eta_max - eta_min) * (1.0 + np.cos(np.pi * u))
    return lr

t = np.arange(0, 101)
lr = warmup_stable_decay(t, eta_max=0.001, warmup=10, stable=70, total=100, eta_min=0.0001)
print(f"start={lr[0]:.6f}, warmup end={lr[10]:.6f}, stable mid={lr[50]:.6f}, final={lr[-1]:.6f}")
check_close("warmup reaches peak", lr[10], 0.001)
check_close("final reaches floor", lr[-1], 0.0001)

fig, ax = plt.subplots()
ax.plot(t, lr, color=COLORS["primary"], label="WSD")
ax.axvline(10, color=COLORS["neutral"], linestyle="--", label="warmup end")
ax.axvline(70, color=COLORS["secondary"], linestyle="--", label="decay start")
ax.set_title("Warmup-stable-decay has explicit phases")
ax.set_xlabel("Optimizer step")
ax.set_ylabel("Learning rate")
ax.legend()
fig.tight_layout()
plt.show()


---

## 4. Classical Decay Curves

Classical schedules differ mainly in how abruptly they reduce $\eta_t$. Plotting them on the same axis reveals their assumptions.


In [ ]:
# === 4.1 Classical schedule comparison ===
t = np.arange(0, 101)
schedules = {
    "constant": constant_schedule(t, eta=0.1),
    "step": step_decay(t, eta=0.1, drop_every=25, gamma=0.5),
    "exponential": exponential_decay(t, eta=0.1, gamma=0.97),
    "linear": polynomial_decay(t, eta=0.1, total_steps=100, power=1.0),
    "quadratic": polynomial_decay(t, eta=0.1, total_steps=100, power=2.0),
}

for name, values in schedules.items():
    print(f"{name:12s} first={values[0]:.5f} mid={values[50]:.5f} final={values[-1]:.5f}")

fig, ax = plt.subplots()
for idx, (name, values) in enumerate(schedules.items()):
    ax.plot(t, values, label=name)
ax.set_title("Classical learning-rate decay schedules")
ax.set_xlabel("Step")
ax.set_ylabel("Learning rate")
ax.legend()
fig.tight_layout()
plt.show()


---

## 5. Step Decay Discontinuities

Step decay can improve final settling, but it creates discontinuities. Those drops are visible in update norms.


In [ ]:
# === 5.1 Step drops produce update drops ===
t = np.arange(0, 81)
theta = 5.0
lambda_value = 4.0
etas = step_decay(t, eta=0.08, drop_every=20, gamma=0.3)
updates = []
losses = []
for eta_t in etas:
    grad = lambda_value * theta
    updates.append(abs(eta_t * grad))
    theta = theta - eta_t * grad
    losses.append(0.5 * lambda_value * theta**2)

print("learning rates at drops:", etas[[0, 19, 20, 39, 40, 60]])
print("update magnitudes near first drop:", np.round(updates[18:23], 6))
check_true("step drop reduces learning rate", etas[20] < etas[19])

fig, ax = plt.subplots()
ax.plot(t, updates, color=COLORS["secondary"], label="update magnitude")
ax.set_title("Step decay creates abrupt update-size drops")
ax.set_xlabel("Step")
ax.set_ylabel(r"$|\eta_t g_t|$")
ax.legend()
fig.tight_layout()
plt.show()


---

## 6. Inverse-Square-Root Transformer Schedule

The Transformer schedule warms up linearly and then decays as $t^{-1/2}$:

$$
\eta_t=d_{\mathrm{model}}^{-1/2}\min(t^{-1/2},tT_{\mathrm{warm}}^{-3/2}).
$$


In [ ]:
# === 6.1 Transformer inverse-square-root schedule ===
def transformer_schedule(step, d_model=512, warmup=4000):
    step = np.asarray(step, dtype=float)
    step = np.maximum(step, 1.0)
    return d_model ** -0.5 * np.minimum(step ** -0.5, step * warmup ** -1.5)

steps = np.arange(1, 20001)
lr = transformer_schedule(steps, d_model=512, warmup=4000)
peak_idx = int(np.argmax(lr))
print(f"peak step = {steps[peak_idx]}")
print(f"peak lr   = {lr[peak_idx]:.8f}")
print(f"lr at 20000 = {lr[-1]:.8f}")
check_true("peak occurs at warmup step", abs(steps[peak_idx] - 4000) <= 1)
check_true("post-warmup decays", lr[-1] < lr[peak_idx])

fig, ax = plt.subplots()
ax.plot(steps, lr, color=COLORS["primary"], label="inverse sqrt with warmup")
ax.axvline(4000, color=COLORS["secondary"], linestyle="--", label="warmup")
ax.set_title("Original Transformer-style learning-rate schedule")
ax.set_xlabel("Optimizer step")
ax.set_ylabel("Learning rate")
ax.legend()
fig.tight_layout()
plt.show()


---

## 7. Linear Warmup

Warmup protects the early training phase by gradually increasing $\eta_t$ to the peak value.


In [ ]:
# === 7.1 Warmup prevents a large first update ===
def linear_warmup(step, eta_max=1e-3, warmup=20):
    step = np.asarray(step, dtype=float)
    return eta_max * np.minimum(step / warmup, 1.0)

steps = np.arange(0, 61)
lr = linear_warmup(steps, eta_max=1e-3, warmup=20)
grad_norm = 100.0 * np.exp(-steps / 30.0) + 5.0
update_no_warmup = 1e-3 * grad_norm
update_warmup = lr * grad_norm

print(f"first update without warmup = {update_no_warmup[0]:.6f}")
print(f"first update with warmup    = {update_warmup[0]:.6f}")
print(f"update at warmup end        = {update_warmup[20]:.6f}")
check_true("warmup reduces first update", update_warmup[1] < update_no_warmup[1])

fig, ax = plt.subplots()
ax.plot(steps, update_no_warmup, color=COLORS["error"], label="no warmup")
ax.plot(steps, update_warmup, color=COLORS["primary"], label="linear warmup")
ax.set_title("Warmup reduces early update magnitude")
ax.set_xlabel("Step")
ax.set_ylabel("Nominal update norm")
ax.legend()
fig.tight_layout()
plt.show()


---

## 8. AdamW Effective Update Scale

The external schedule multiplies the AdamW normalized update. Warmup still matters even when the optimizer is adaptive.


In [ ]:
# === 8.1 AdamW with and without warmup on a gradient sequence ===
grads = np.array([8.0, 6.0, 3.0, 2.0, 1.5, 1.0, 0.7, 0.4])
beta1, beta2, eps = 0.9, 0.99, 1e-8

def adam_normalized_steps(grads):
    m = 0.0
    v = 0.0
    normalized = []
    for t, g in enumerate(grads, start=1):
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g * g
        mhat = m / (1 - beta1 ** t)
        vhat = v / (1 - beta2 ** t)
        normalized.append(mhat / (np.sqrt(vhat) + eps))
    return np.array(normalized)

norm_step = adam_normalized_steps(grads)
eta_const = np.full_like(norm_step, 1e-3)
eta_warm = linear_warmup(np.arange(1, len(norm_step) + 1), eta_max=1e-3, warmup=5)

print("normalized Adam steps:", np.round(norm_step, 4))
print("constant LR updates: ", np.round(eta_const * norm_step, 6))
print("warmup LR updates:   ", np.round(eta_warm * norm_step, 6))
check_true("warmup starts smaller than constant LR", eta_warm[0] < eta_const[0])


---

## 9. Batch-Size Scaling

Peak learning rate is meaningful only relative to batch size. Linear scaling and square-root scaling are useful heuristics, not laws.


In [ ]:
# === 9.1 Linear versus square-root batch scaling ===
base_batch = 256
base_lr = 1e-3
batches = np.array([256, 512, 1024, 2048, 4096, 8192])
linear_lr = base_lr * batches / base_batch
sqrt_lr = base_lr * np.sqrt(batches / base_batch)

print("batch  linear_lr  sqrt_lr")
for b, l1, l2 in zip(batches, linear_lr, sqrt_lr):
    print(f"{b:5d}  {l1:9.6f}  {l2:7.6f}")
check_true("linear scaling is more aggressive", linear_lr[-1] > sqrt_lr[-1])

fig, ax = plt.subplots()
ax.plot(batches, linear_lr, marker="o", color=COLORS["primary"], label="linear scaling")
ax.plot(batches, sqrt_lr, marker="o", color=COLORS["secondary"], label="sqrt scaling")
ax.set_title("Batch-size scaling rules for peak learning rate")
ax.set_xlabel("Global batch size")
ax.set_ylabel("Peak learning rate")
ax.legend()
fig.tight_layout()
plt.show()


---

## 10. Cosine Annealing

Cosine annealing smoothly maps a fixed training horizon to a final low learning rate.


In [ ]:
# === 10.1 Cosine annealing with a floor ===
def cosine_decay(step, eta_max=1.0, total=100, eta_min=0.0):
    step = np.asarray(step, dtype=float)
    u = np.clip(step / total, 0.0, 1.0)
    return eta_min + 0.5 * (eta_max - eta_min) * (1.0 + np.cos(np.pi * u))

steps = np.arange(0, 101)
lr = cosine_decay(steps, eta_max=0.1, total=100, eta_min=0.001)
print(f"start={lr[0]:.6f}, mid={lr[50]:.6f}, final={lr[100]:.6f}")
check_close("cosine starts at peak", lr[0], 0.1)
check_close("cosine ends at floor", lr[-1], 0.001)

fig, ax = plt.subplots()
ax.plot(steps, lr, color=COLORS["primary"], label="cosine")
ax.set_title("Cosine annealing with minimum learning-rate floor")
ax.set_xlabel("Step")
ax.set_ylabel("Learning rate")
ax.legend()
fig.tight_layout()
plt.show()


---

## 11. Learning-Rate Floors

A floor $\eta_{\min}=\alpha\eta_{\max}$ preserves late adaptation. Decay-to-zero gives a stronger final cooldown but less flexibility.


In [ ]:
# === 11.1 Comparing cosine floors ===
steps = np.arange(0, 101)
floors = [0.0, 0.01, 0.1]
fig, ax = plt.subplots()
for alpha in floors:
    values = cosine_decay(steps, eta_max=1.0, total=100, eta_min=alpha)
    ax.plot(steps, values, label=f"floor ratio {alpha}")
    print(f"floor ratio {alpha:4.2f}: final LR = {values[-1]:.3f}")
ax.set_title("Cosine floors control late adaptation")
ax.set_xlabel("Step")
ax.set_ylabel("Learning-rate multiplier")
ax.legend()
fig.tight_layout()
plt.show()
check_true("0.1 floor ends above 0.01 floor", cosine_decay(np.array([100]), eta_min=0.1)[0] > cosine_decay(np.array([100]), eta_min=0.01)[0])


---

## 12. SGDR Warm Restarts

Warm restarts reset the cycle clock while keeping parameters. The learning rate jumps up, but the model is not reinitialized.


In [ ]:
# === 12.1 SGDR cycle clock ===
def sgdr_schedule(steps, eta_max=0.1, eta_min=0.0, first_cycle=20, cycle_mult=2):
    values = []
    cycle_len = first_cycle
    cycle_start = 0
    for t in steps:
        while t >= cycle_start + cycle_len:
            cycle_start += cycle_len
            cycle_len *= cycle_mult
        t_cur = t - cycle_start
        values.append(eta_min + 0.5 * (eta_max - eta_min) * (1 + np.cos(np.pi * t_cur / cycle_len)))
    return np.array(values)

steps = np.arange(0, 121)
lr = sgdr_schedule(steps, eta_max=0.1, eta_min=0.005, first_cycle=20, cycle_mult=2)
print("LR at restart-like steps:", np.round(lr[[0, 19, 20, 59, 60]], 6))
check_true("LR jumps up after first restart", lr[20] > lr[19])

fig, ax = plt.subplots()
ax.plot(steps, lr, color=COLORS["primary"], label="SGDR")
ax.set_title("Cosine annealing with warm restarts")
ax.set_xlabel("Step")
ax.set_ylabel("Learning rate")
ax.legend()
fig.tight_layout()
plt.show()


---

## 13. Cyclical Learning Rates

Cyclical schedules repeatedly move between lower and upper learning-rate bounds.


In [ ]:
# === 13.1 Triangular cyclic schedule ===
def triangular_cycle(steps, base_lr=0.001, max_lr=0.01, step_size_up=20, mode="triangular", gamma=0.999):
    steps = np.asarray(steps)
    cycle = np.floor(1 + steps / (2 * step_size_up))
    x = np.abs(steps / step_size_up - 2 * cycle + 1)
    scale = np.maximum(0.0, 1.0 - x)
    if mode == "triangular2":
        scale = scale / (2 ** (cycle - 1))
    elif mode == "exp_range":
        scale = scale * gamma ** steps
    return base_lr + (max_lr - base_lr) * scale

steps = np.arange(0, 121)
tri = triangular_cycle(steps, mode="triangular")
tri2 = triangular_cycle(steps, mode="triangular2")
expr = triangular_cycle(steps, mode="exp_range", gamma=0.98)
print("first cycle peak triangular:", tri[20])
print("second cycle peak triangular2:", tri2[60])
check_true("triangular2 second peak is smaller", tri2[60] < tri[60])

fig, ax = plt.subplots()
ax.plot(steps, tri, label="triangular", color=COLORS["primary"])
ax.plot(steps, tri2, label="triangular2", color=COLORS["secondary"])
ax.plot(steps, expr, label="exp_range", color=COLORS["tertiary"])
ax.set_title("Cyclical learning-rate variants")
ax.set_xlabel("Step")
ax.set_ylabel("Learning rate")
ax.legend()
fig.tight_layout()
plt.show()


---

## 14. LR Range Test

An LR range test increases $\eta_t$ over a short run and looks for the region where loss decreases before divergence.


In [ ]:
# === 14.1 Synthetic LR range test ===
steps = np.arange(0, 100)
test_lr = np.geomspace(1e-5, 1.0, len(steps))
noise = 0.03 * np.random.randn(len(steps))
synthetic_loss = 2.0 - 0.6 * np.log10(test_lr / 1e-5 + 1) + 25.0 * np.maximum(test_lr - 0.2, 0) ** 2 + noise
best_idx = int(np.argmin(synthetic_loss))
suggested_low = test_lr[max(best_idx // 4, 1)]
suggested_high = test_lr[best_idx]

print(f"best observed lr   = {test_lr[best_idx]:.5f}")
print(f"suggested low/high = {suggested_low:.5f}, {suggested_high:.5f}")
check_true("suggested bounds are ordered", suggested_low < suggested_high)

fig, ax = plt.subplots()
ax.semilogx(test_lr, synthetic_loss, color=COLORS["primary"], label="range-test loss")
ax.axvline(suggested_low, color=COLORS["secondary"], linestyle="--", label="low bound")
ax.axvline(suggested_high, color=COLORS["error"], linestyle="--", label="high bound")
ax.set_title("Synthetic learning-rate range test")
ax.set_xlabel("Learning rate")
ax.set_ylabel("Loss")
ax.legend()
fig.tight_layout()
plt.show()


---

## 15. One-Cycle Policy

One-cycle increases learning rate, then decays to a very low final value. Momentum is often cycled inversely.


In [ ]:
# === 15.1 One-cycle LR and inverse momentum ===
def one_cycle(steps, max_lr=0.01, total=100, pct_start=0.3, div_factor=25, final_div_factor=1e4):
    steps = np.asarray(steps, dtype=float)
    initial_lr = max_lr / div_factor
    final_lr = initial_lr / final_div_factor
    up_steps = pct_start * total
    lr = np.empty_like(steps)
    up = steps <= up_steps
    lr[up] = initial_lr + (max_lr - initial_lr) * (steps[up] / max(up_steps, 1))
    down_u = np.clip((steps[~up] - up_steps) / max(total - up_steps, 1), 0.0, 1.0)
    lr[~up] = final_lr + 0.5 * (max_lr - final_lr) * (1.0 + np.cos(np.pi * down_u))
    return lr

def inverse_momentum(lr, lr_min, lr_max, mom_min=0.85, mom_max=0.95):
    scale = (lr - lr_min) / max(lr_max - lr_min, 1e-12)
    return mom_max - scale * (mom_max - mom_min)

steps = np.arange(0, 101)
lr = one_cycle(steps, max_lr=0.01, total=100, pct_start=0.3)
mom = inverse_momentum(lr, lr.min(), lr.max())
print(f"initial lr={lr[0]:.8f}, max lr={lr.max():.5f}, final lr={lr[-1]:.10f}")
print(f"momentum at max lr={mom[np.argmax(lr)]:.4f}, initial momentum={mom[0]:.4f}")
check_true("one-cycle ends below initial lr", lr[-1] < lr[0])
check_true("momentum is low at max LR", mom[np.argmax(lr)] < mom[0])

fig, ax1 = plt.subplots()
ax1.plot(steps, lr, color=COLORS["primary"], label="learning rate")
ax1.set_xlabel("Step")
ax1.set_ylabel("Learning rate")
ax2 = ax1.twinx()
ax2.plot(steps, mom, color=COLORS["secondary"], label="momentum")
ax2.set_ylabel("Momentum")
ax1.set_title("One-cycle policy with inverse momentum")
fig.tight_layout()
plt.show()


---

## 16. Large Learning Rates as Regularization

A large learning rate can prevent overly precise fitting by injecting optimization noise. This is the intuition behind super-convergence, but it is not safe in every regime.


In [ ]:
# === 16.1 Large-LR noise intuition in a one-dimensional problem ===
np.random.seed(42)
steps = 120
theta_small = 3.0
theta_large = 3.0
small_path = []
large_path = []
for t in range(steps):
    noise = 0.2 * np.random.randn()
    grad_small = theta_small + noise
    grad_large = theta_large + noise
    eta_small = 0.03
    eta_large = one_cycle(np.array([t]), max_lr=0.25, total=steps, pct_start=0.25)[0]
    theta_small -= eta_small * grad_small
    theta_large -= eta_large * grad_large
    small_path.append(theta_small)
    large_path.append(theta_large)

print(f"small constant LR final theta = {theta_small:.5f}")
print(f"large one-cycle final theta   = {theta_large:.5f}")
check_true("both runs remain finite", np.isfinite(theta_small) and np.isfinite(theta_large))

fig, ax = plt.subplots()
ax.plot(small_path, color=COLORS["primary"], label="small constant LR")
ax.plot(large_path, color=COLORS["secondary"], label="large one-cycle LR")
ax.set_title("Large LR creates a different training path")
ax.set_xlabel("Step")
ax.set_ylabel(r"$\theta_t$")
ax.legend()
fig.tight_layout()
plt.show()


---

## 17. Warmup-Stable-Decay

WSD separates the long main training branch from the final decay branch.


In [ ]:
# === 17.1 WSD schedule ===
steps = np.arange(0, 151)
wsd_lr = warmup_stable_decay(steps, eta_max=0.001, warmup=10, stable=110, total=150, eta_min=0.00005)
cos_lr = np.concatenate([
    linear_warmup(np.arange(0, 10), eta_max=0.001, warmup=10),
    cosine_decay(np.arange(0, 141), eta_max=0.001, total=140, eta_min=0.00005),
])
print(f"WSD lr at step 75    = {wsd_lr[75]:.6f}")
print(f"Cosine lr at step 75 = {cos_lr[75]:.6f}")
print(f"WSD final lr         = {wsd_lr[-1]:.6f}")
check_true("WSD keeps higher LR during stable phase", wsd_lr[75] > cos_lr[75])

fig, ax = plt.subplots()
ax.plot(steps, wsd_lr, color=COLORS["primary"], label="WSD")
ax.plot(steps, cos_lr[:len(steps)], color=COLORS["secondary"], label="warmup cosine")
ax.set_title("WSD holds a high learning rate longer than cosine")
ax.set_xlabel("Step")
ax.set_ylabel("Learning rate")
ax.legend()
fig.tight_layout()
plt.show()


---

## 18. WSD Branching

In budget-uncertain pretraining, selected checkpoints on the stable branch can spawn final decay branches.


In [ ]:
# === 18.1 Branch schedules from one main WSD branch ===
main_steps = np.arange(0, 301)
branch_points = [120, 200, 280]
decay_len = 40
eta_max = 0.001
eta_min = 0.00005

print("branch point -> final branch LR")
fig, ax = plt.subplots()
ax.plot(main_steps, warmup_stable_decay(main_steps, eta_max=eta_max, warmup=20, stable=10_000, total=10_000, eta_min=eta_min),
        color=COLORS["neutral"], label="main stable branch")
for j, start in enumerate(branch_points, start=1):
    local = np.arange(0, decay_len + 1)
    decay = cosine_decay(local, eta_max=eta_max, total=decay_len, eta_min=eta_min)
    ax.plot(start + local, decay, label=f"decay branch {j}")
    print(f"{start:4d} -> {decay[-1]:.6f}")
    check_close(f"branch {j} reaches floor", decay[-1], eta_min)

ax.set_title("WSD can branch multiple final decays from one run")
ax.set_xlabel("Global step")
ax.set_ylabel("Learning rate")
ax.legend()
fig.tight_layout()
plt.show()


---

## 19. River-Valley Toy Landscape

The river-valley intuition separates an oscillatory sharp direction from a slower progress direction.


In [ ]:
# === 19.1 Toy river-valley loss under cosine and WSD ===
def river_grad(theta):
    u, v = theta
    # Sharp direction u, shallow curved direction v.
    return np.array([30.0 * u, 0.1 * (v - 3.0)])

def run_schedule(lrs, theta=np.array([1.0, 0.0])):
    theta = theta.astype(float).copy()
    path = []
    losses = []
    for eta in lrs:
        theta = theta - eta * river_grad(theta)
        u, v = theta
        loss = 15.0 * u**2 + 0.05 * (v - 3.0)**2
        path.append(theta.copy())
        losses.append(loss)
    return np.array(path), np.array(losses)

steps = np.arange(0, 180)
cos_lrs = cosine_decay(steps, eta_max=0.055, total=179, eta_min=0.002)
wsd_lrs = warmup_stable_decay(steps, eta_max=0.055, warmup=10, stable=140, total=179, eta_min=0.002)
cos_path, cos_loss = run_schedule(cos_lrs)
wsd_path, wsd_loss = run_schedule(wsd_lrs)

print(f"cosine final loss = {cos_loss[-1]:.6f}")
print(f"WSD final loss    = {wsd_loss[-1]:.6f}")
check_true("losses are finite", np.all(np.isfinite(cos_loss)) and np.all(np.isfinite(wsd_loss)))

fig, ax = plt.subplots()
ax.plot(cos_loss, color=COLORS["secondary"], label="cosine")
ax.plot(wsd_loss, color=COLORS["primary"], label="WSD")
ax.set_title("Toy river-valley loss curves")
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.legend()
fig.tight_layout()
plt.show()


---

## 20. Gradient Accumulation Indexing

Schedulers should usually advance on optimizer steps, not microbatches.


In [ ]:
# === 20.1 Gradient accumulation schedule bug ===
microbatches = 800
accum = 8
intended_warmup_optimizer_steps = 100
optimizer_steps = microbatches // accum

wrong_effective_warmup = intended_warmup_optimizer_steps / accum
print(f"microbatches: {microbatches}")
print(f"gradient accumulation: {accum}")
print(f"optimizer steps: {optimizer_steps}")
print(f"intended warmup optimizer steps: {intended_warmup_optimizer_steps}")
print(f"actual warmup if scheduler steps every microbatch: {wrong_effective_warmup:.1f} optimizer steps")
check_close("wrong warmup is divided by accumulation", wrong_effective_warmup, 12.5)


---

## 21. Resume-State Bugs

If a scheduler restarts from zero after checkpoint resume, the learning rate no longer matches the training state.


In [ ]:
# === 21.1 Correct versus incorrect resume ===
total = 1000
resume_step = 650
future = np.arange(resume_step, resume_step + 80)
correct = cosine_decay(future, eta_max=1e-3, total=total, eta_min=1e-4)
wrong = cosine_decay(np.arange(0, len(future)), eta_max=1e-3, total=total, eta_min=1e-4)

print(f"correct resume lr at step {resume_step}: {correct[0]:.7f}")
print(f"wrong restarted lr:              {wrong[0]:.7f}")
print(f"ratio wrong/correct:             {wrong[0] / correct[0]:.2f}")
check_true("wrong resume LR is much larger", wrong[0] > 2.0 * correct[0])

fig, ax = plt.subplots()
ax.plot(future, correct, color=COLORS["primary"], label="correct scheduler state")
ax.plot(future, wrong, color=COLORS["error"], label="scheduler restarted at 0")
ax.set_title("Scheduler resume state changes the LR")
ax.set_xlabel("Global step")
ax.set_ylabel("Learning rate")
ax.legend()
fig.tight_layout()
plt.show()


---

## 22. API Equivalence

Most library schedulers are wrappers around simple mathematical functions. Test the function directly before trusting a training run.


In [ ]:
# === 22.1 Lambda-style warmup cosine schedule ===
def warmup_cosine_multiplier(step, warmup_steps, total_steps, min_ratio=0.0):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    progress = min(max(progress, 0.0), 1.0)
    return min_ratio + 0.5 * (1.0 - min_ratio) * (1.0 + np.cos(np.pi * progress))

for s in [0, 5, 10, 50, 100]:
    print(f"step {s:3d}: multiplier={warmup_cosine_multiplier(s, 10, 100, min_ratio=0.1):.6f}")

check_close("start multiplier", warmup_cosine_multiplier(0, 10, 100, min_ratio=0.1), 0.0)
check_close("warmup end multiplier", warmup_cosine_multiplier(10, 10, 100, min_ratio=0.1), 1.0)
check_close("final multiplier", warmup_cosine_multiplier(100, 10, 100, min_ratio=0.1), 0.1)


---

## 23. Diagnostics

Learning rate values alone are insufficient. The update ratio reveals how much the model actually moves.


In [ ]:
# === 23.1 Synthetic training diagnostics ===
steps = np.arange(0, 160)
lr = warmup_stable_decay(steps, eta_max=2e-3, warmup=15, stable=110, total=159, eta_min=2e-4)
grad_norm = 8.0 * np.exp(-steps / 100) + 0.5 + 0.2 * np.sin(steps / 8)
param_norm = 100.0 + 0.03 * steps
update_norm = lr * grad_norm * 20.0
update_ratio = update_norm / param_norm
clip_threshold = 0.0013
clip_events = update_ratio > clip_threshold

print(f"max update ratio = {update_ratio.max():.6f}")
print(f"clip event count = {clip_events.sum()}")
print(f"final update ratio = {update_ratio[-1]:.6f}")
check_true("decay lowers final update ratio", update_ratio[-1] < update_ratio[80])

fig, ax = plt.subplots()
ax.plot(steps, update_ratio, color=COLORS["primary"], label="update ratio")
ax.axhline(clip_threshold, color=COLORS["error"], linestyle="--", label="diagnostic threshold")
ax.set_title("Update ratio diagnoses schedule aggressiveness")
ax.set_xlabel("Step")
ax.set_ylabel(r"$\Vert \Delta\theta_t \Vert_2 / \Vert \theta_t \Vert_2$")
ax.legend()
fig.tight_layout()
plt.show()


---

## 24. Schedule Selection Heuristic

A schedule recommendation is a hypothesis. The diagnostics decide whether it survives contact with training.


In [ ]:
# === 24.1 Simple schedule recommender ===
def recommend_schedule(setting, fixed_budget=True, budget_uncertain=False, small_model=False):
    if budget_uncertain:
        return "warmup-stable-decay"
    if small_model:
        return "one-cycle or cosine after LR range test"
    if setting == "fine-tune":
        return "linear warmup + linear or cosine decay"
    if fixed_budget:
        return "linear warmup + cosine decay"
    return "constant with warmup or WSD"

cases = [
    ("LLM pretraining", dict(setting="pretrain", fixed_budget=True)),
    ("continual LLM", dict(setting="pretrain", budget_uncertain=True)),
    ("BERT fine-tune", dict(setting="fine-tune")),
    ("small vision", dict(setting="supervised", small_model=True)),
]
for name, kwargs in cases:
    print(f"{name:16s} -> {recommend_schedule(**kwargs)}")
check_true("budget uncertainty selects WSD", recommend_schedule("pretrain", budget_uncertain=True) == "warmup-stable-decay")


---

## 25. Compare Schedule Values

A compact table catches many implementation errors.


In [ ]:
# === 25.1 Boundary-value table ===
probe = np.array([0, 10, 25, 50, 75, 100])
table = {
    "step": step_decay(probe, eta=1.0, drop_every=25, gamma=0.5),
    "linear": polynomial_decay(probe, eta=1.0, total_steps=100, power=1.0),
    "cosine": cosine_decay(probe, eta_max=1.0, total=100, eta_min=0.0),
    "wsd": warmup_stable_decay(probe, eta_max=1.0, warmup=10, stable=70, total=100, eta_min=0.0),
}
print("step | " + " | ".join(f"{k:>8s}" for k in table))
for i, s in enumerate(probe):
    row = " | ".join(f"{table[k][i]:8.4f}" for k in table)
    print(f"{s:4d} | {row}")
check_close("cosine final is zero", table["cosine"][-1], 0.0)
check_close("WSD warmup end is peak", table["wsd"][1], 1.0)


---

## 26. Mini Stochastic Training Experiment

The same optimizer can reach different final states under different $\eta_t$ policies.


In [ ]:
# === 26.1 Schedules on a stochastic quadratic ===
def train_quadratic(lr_values, seed=0):
    rng = np.random.default_rng(seed)
    theta = np.array([4.0, -3.0])
    H = np.diag([1.0, 20.0])
    losses = []
    for eta in lr_values:
        grad = H @ theta + rng.normal(scale=0.25, size=2)
        theta = theta - eta * grad
        losses.append(0.5 * theta @ H @ theta)
    return np.array(losses), theta

steps = np.arange(0, 160)
constant_lr = constant_schedule(steps, eta=0.04)
cos_lr = cosine_decay(steps, eta_max=0.04, total=159, eta_min=0.002)
wsd_lr = warmup_stable_decay(steps, eta_max=0.04, warmup=10, stable=120, total=159, eta_min=0.002)

loss_const, theta_const = train_quadratic(constant_lr, seed=1)
loss_cos, theta_cos = train_quadratic(cos_lr, seed=1)
loss_wsd, theta_wsd = train_quadratic(wsd_lr, seed=1)

print(f"constant final loss = {loss_const[-1]:.6f}")
print(f"cosine final loss   = {loss_cos[-1]:.6f}")
print(f"WSD final loss      = {loss_wsd[-1]:.6f}")
check_true("all losses finite", np.all(np.isfinite(loss_const)) and np.all(np.isfinite(loss_cos)) and np.all(np.isfinite(loss_wsd)))

fig, ax = plt.subplots()
ax.semilogy(loss_const, color=COLORS["neutral"], label="constant")
ax.semilogy(loss_cos, color=COLORS["secondary"], label="cosine")
ax.semilogy(loss_wsd, color=COLORS["primary"], label="WSD")
ax.set_title("Schedule choice changes stochastic optimization")
ax.set_xlabel("Step")
ax.set_ylabel("Quadratic loss")
ax.legend()
fig.tight_layout()
plt.show()


---

## Recap

You have implemented the main learning-rate schedule families:

- fixed and monotone decay schedules,
- linear warmup,
- inverse-square-root transformer scheduling,
- cosine annealing and floors,
- warm restarts,
- cyclic and one-cycle policies,
- warmup-stable-decay,
- scheduler-state and gradient-accumulation diagnostics.

The main operational lesson is simple: always plot $\eta_t$, always save scheduler state, and always interpret the schedule through update norms and validation behavior.
